Import libraries

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "runs" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Reports directory:", REPORT_DIR)

Project root: c:\Users\yatha\Desktop\personalized-recommendation-engine
Processed data directory: c:\Users\yatha\Desktop\personalized-recommendation-engine\data\processed
Reports directory: c:\Users\yatha\Desktop\personalized-recommendation-engine\runs\reports


Load datasets

In [3]:
candidate_df = pd.read_csv(
    REPORT_DIR / "faiss_candidate_recommendations.csv"
)

train_df = pd.read_csv(
    PROCESSED_DIR / "train.csv"
)

validation_df = pd.read_csv(
    PROCESSED_DIR / "validation.csv"
)

test_df = pd.read_csv(
    PROCESSED_DIR / "test.csv"
)

movies_df = pd.read_csv(
    PROJECT_ROOT / "data" / "ml-latest-small" / "movies.csv"
)

movie_mapping = pd.read_csv(
    PROCESSED_DIR / "movie_mapping.csv"
)

print("Candidate data:", candidate_df.shape)
print("Train data:", train_df.shape)
print("Validation data:", validation_df.shape)
print("Test data:", test_df.shape)
print("Movies data:", movies_df.shape)

display(candidate_df.head())

Candidate data: (61000, 6)
Train data: (80419, 6)
Validation data: (9818, 6)
Test data: (10599, 6)
Movies data: (9742, 3)


,user_idx,rank,movie_idx,similarity_score,title,genres
0,0,1,2173,-0.171012,"Mystery, Alaska (1999)",Comedy|Drama
1,0,2,1288,-0.182374,Scream 2 (1997),Comedy|Horror|Mystery|Thriller
2,0,3,1109,-0.190847,Prefontaine (1997),Drama
3,0,4,1192,-0.195870,Cop Land (1997),Action|Crime|Drama|Thriller
4,0,5,1246,-0.199653,Swept from the Sea (1997),Drama|Romance


Inspect columns and verify candidate data

In [4]:
print("Candidate columns:")
print(candidate_df.columns.tolist())

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

display(candidate_df[[
    "user_idx",
    "rank",
    "movie_idx",
    "similarity_score",
    "title",
    "genres"
]].head(10))

Candidate columns:
['user_idx', 'rank', 'movie_idx', 'similarity_score', 'title', 'genres']

Train columns:
['userId', 'movieId', 'rating', 'timestamp', 'user_idx', 'movie_idx']

Validation columns:
['userId', 'movieId', 'rating', 'timestamp', 'user_idx', 'movie_idx']


,user_idx,rank,movie_idx,similarity_score,title,genres
0,0,1,2173,-0.171012,"Mystery, Alaska (1999)",Comedy|Drama
1,0,2,1288,-0.182374,Scream 2 (1997),Comedy|Horror|Mystery|Thriller
2,0,3,1109,-0.190847,Prefontaine (1997),Drama
3,0,4,1192,-0.195870,Cop Land (1997),Action|Crime|Drama|Thriller
4,0,5,1246,-0.199653,Swept from the Sea (1997),Drama|Romance
5,0,6,3661,-0.200133,Dragonslayer (1981),Action|Adventure|Fantasy
6,0,7,4045,-0.200212,Galaxy of Terror (Quest) (1981),Action|Horror|Mystery|Sci-Fi
7,0,8,3723,-0.201721,40 Days and 40 Nights (2002),Comedy|Romance
8,0,9,3702,-0.202818,Hart's War (2002),Drama|War
9,0,10,2131,-0.203948,Outside Providence (1999),Comedy


Create relevance labels

In [5]:
RELEVANCE_THRESHOLD = 4.0

validation_positive_df = validation_df[
    validation_df["rating"] >= RELEVANCE_THRESHOLD
].copy()

validation_positive_pairs = set(
    zip(
        validation_positive_df["user_idx"],
        validation_positive_df["movie_idx"]
    )
)

candidate_df["label"] = [
    int((user_idx, movie_idx) in validation_positive_pairs)
    for user_idx, movie_idx in zip(
        candidate_df["user_idx"],
        candidate_df["movie_idx"]
    )
]

print("Positive validation interactions:",
      len(validation_positive_pairs))

print("Positive candidate labels:",
      candidate_df["label"].sum())

print("Negative candidate labels:",
      (candidate_df["label"] == 0).sum())

display(candidate_df.head())

Positive validation interactions: 4451
Positive candidate labels: 377
Negative candidate labels: 60623


,user_idx,rank,movie_idx,similarity_score,title,genres,label
0,0,1,2173,-0.171012,"Mystery, Alaska (1999)",Comedy|Drama,0
1,0,2,1288,-0.182374,Scream 2 (1997),Comedy|Horror|Mystery|Thriller,0
2,0,3,1109,-0.190847,Prefontaine (1997),Drama,0
3,0,4,1192,-0.195870,Cop Land (1997),Action|Crime|Drama|Thriller,0
4,0,5,1246,-0.199653,Swept from the Sea (1997),Drama|Romance,0


Generate user features

In [6]:
user_features = (
    train_df.groupby("user_idx")
    .agg(
        user_avg_rating=("rating", "mean"),
        user_rating_count=("rating", "count")
    )
    .reset_index()
)

user_features["user_rating_count"] = np.log1p(
    user_features["user_rating_count"]
)

display(user_features.head())

,user_idx,user_avg_rating,user_rating_count
0,0,4.367568,5.225747
1,1,3.847826,3.178054
2,2,2.935484,3.465736
3,3,3.627907,5.153292
4,4,3.600000,3.583519


Generate movie popularity features

In [7]:
movie_features = (
    train_df.groupby("movie_idx")
    .agg(
        movie_rating_count=("rating", "count"),
        movie_avg_rating=("rating", "mean")
    )
    .reset_index()
)

movie_features["movie_rating_count"] = np.log1p(
    movie_features["movie_rating_count"]
)

display(movie_features.head())

,movie_idx,movie_rating_count,movie_avg_rating
0,0,5.308268,3.915423
1,1,4.532599,3.413043
2,2,3.737670,3.280488
3,3,1.609438,2.125000
4,4,3.688879,3.025641


Encode movie genres

In [8]:
candidate_df["genres_list"] = candidate_df["genres"].fillna(
    "(no genres listed)"
).apply(
    lambda x: x.split("|")
)

genre_encoder = MultiLabelBinarizer()

genre_features = genre_encoder.fit_transform(
    candidate_df["genres_list"]
)

genre_feature_names = [
    f"genre_{genre}" for genre in genre_encoder.classes_
]

genre_df = pd.DataFrame(
    genre_features,
    columns=genre_feature_names,
    index=candidate_df.index
)

print("Number of genre features:", len(genre_feature_names))

display(genre_df.head())

Number of genre features: 20


,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0
2,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0
4,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0


Combine all ranking features

In [9]:
ranking_data = candidate_df.merge(
    user_features,
    on="user_idx",
    how="left"
)

ranking_data = ranking_data.merge(
    movie_features,
    on="movie_idx",
    how="left"
)

ranking_data = pd.concat(
    [ranking_data, genre_df],
    axis=1
)

numeric_feature_columns = [
    "similarity_score",
    "user_avg_rating",
    "user_rating_count",
    "movie_rating_count",
    "movie_avg_rating"
] + genre_feature_names

ranking_data[numeric_feature_columns] = (
    ranking_data[numeric_feature_columns].fillna(0)
)

print("Ranking dataset shape:", ranking_data.shape)

display(
    ranking_data[
        ["user_idx", "movie_idx", "label"] +
        numeric_feature_columns[:5]
    ].head()
)

Ranking dataset shape: (61000, 32)


,user_idx,movie_idx,label,similarity_score,user_avg_rating,user_rating_count,movie_rating_count,movie_avg_rating
0,0,2173,0,-0.171012,4.367568,5.225747,1.945910,3.916667
1,0,1288,0,-0.182374,4.367568,5.225747,3.332205,2.907407
2,0,1109,0,-0.190847,4.367568,5.225747,1.098612,4.000000
3,0,1192,0,-0.195870,4.367568,5.225747,2.564949,3.333333
4,0,1246,0,-0.199653,4.367568,5.225747,0.693147,4.000000


Train the Random Forest ranker

In [10]:
X_train = ranking_data[numeric_feature_columns]

y_train = ranking_data["label"]

ranker = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

ranker.fit(X_train, y_train)

print("Random Forest ranking model trained successfully.")

Random Forest ranking model trained successfully.


Predict relevance scores

In [11]:
ranking_data["ranking_score"] = ranker.predict_proba(
    X_train
)[:, 1]

display(
    ranking_data[
        [
            "user_idx",
            "rank",
            "title",
            "similarity_score",
            "label",
            "ranking_score"
        ]
    ].sort_values(
        ["user_idx", "ranking_score"],
        ascending=[True, False]
    ).head(20)
)

,user_idx,rank,title,similarity_score,label,ranking_score
31,0,32,Scream 3 (2000),-0.221182,1,0.613313
55,0,56,"Texas Chainsaw Massacre, The (1974)",-0.237648,1,0.485186
1,0,2,Scream 2 (1997),-0.182374,0,0.470660
5,0,6,Dragonslayer (1981),-0.200133,0,0.356449
56,0,57,Shallow Hal (2001),-0.237681,0,0.293128
83,0,84,Creepshow (1982),-0.249053,0,0.281191
58,0,59,"Dark Crystal, The (1982)",-0.238377,0,0.269246
34,0,35,"Addams Family, The (1991)",-0.226755,0,0.258124
22,0,23,Keeping the Faith (2000),-0.213885,0,0.235307
24,0,25,Escape from L.A. (1996),-0.215423,0,0.225672


Generate the final ranked Top-10 recommendations

In [13]:
ranked_recommendations = (
    ranking_data
    .sort_values(
        ["user_idx", "ranking_score"],
        ascending=[True, False]
    )
    .groupby("user_idx")
    .head(10)
    .copy()
)

ranked_recommendations["final_rank"] = (
    ranked_recommendations
    .groupby("user_idx")
    .cumcount() + 1
)

print("Final recommendations generated.")
print("Total recommendations:", len(ranked_recommendations))
print("Users:", ranked_recommendations["user_idx"].nunique())

display(
    ranked_recommendations[
        [
            "user_idx",
            "final_rank",
            "title",
            "genres",
            "ranking_score"
        ]
    ].head(20)
)

Final recommendations generated.
Total recommendations: 6100
Users: 610


,user_idx,final_rank,title,genres,ranking_score
31,0,1,Scream 3 (2000),Comedy|Horror|Mystery|Thriller,0.613313
55,0,2,"Texas Chainsaw Massacre, The (1974)",Horror,0.485186
1,0,3,Scream 2 (1997),Comedy|Horror|Mystery|Thriller,0.470660
5,0,4,Dragonslayer (1981),Action|Adventure|Fantasy,0.356449
56,0,5,Shallow Hal (2001),Comedy|Fantasy|Romance,0.293128
83,0,6,Creepshow (1982),Horror,0.281191
58,0,7,"Dark Crystal, The (1982)",Adventure|Fantasy,0.269246
34,0,8,"Addams Family, The (1991)",Children|Comedy|Fantasy,0.258124
22,0,9,Keeping the Faith (2000),Comedy|Drama|Romance,0.235307
24,0,10,Escape from L.A. (1996),Action|Adventure|Sci-Fi|Thriller,0.225672


Evaluate ranking performance on the test set

In [15]:
test_positive_df = test_df[
    test_df["rating"] >= RELEVANCE_THRESHOLD
].copy()

test_relevant_movies = (
    test_positive_df
    .groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)

def recall_at_k(recommended, relevant, k=10):
    if len(relevant) == 0:
        return 0.0

    hits = len(set(recommended[:k]) & relevant)

    return hits / len(relevant)


def precision_at_k(recommended, relevant, k=10):
    hits = len(set(recommended[:k]) & relevant)

    return hits / k


def ndcg_at_k(recommended, relevant, k=10):
    dcg = sum(
        1 / np.log2(rank + 1)
        for rank, movie_idx in enumerate(recommended[:k], start=1)
        if movie_idx in relevant
    )

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0

Calculate final metrics

In [16]:
test_recall_scores = []
test_precision_scores = []
test_ndcg_scores = []

for user_idx, relevant in test_relevant_movies.items():

    user_recommendations = (
        ranked_recommendations[
            ranked_recommendations["user_idx"] == user_idx
        ]
        .sort_values("final_rank")["movie_idx"]
        .tolist()
    )

    test_recall_scores.append(
        recall_at_k(user_recommendations, relevant, 10)
    )

    test_precision_scores.append(
        precision_at_k(user_recommendations, relevant, 10)
    )

    test_ndcg_scores.append(
        ndcg_at_k(user_recommendations, relevant, 10)
    )

ranking_recall = np.mean(test_recall_scores)
ranking_precision = np.mean(test_precision_scores)
ranking_ndcg = np.mean(test_ndcg_scores)

print("Random Forest Ranking Model — Test Results")
print("-----------------------------------------")
print(f"Recall@10:    {ranking_recall:.4f}")
print(f"Precision@10: {ranking_precision:.4f}")
print(f"NDCG@10:      {ranking_ndcg:.4f}")
print("Evaluated users:", len(test_relevant_movies))

Random Forest Ranking Model — Test Results
-----------------------------------------
Recall@10:    0.0422
Precision@10: 0.0190
NDCG@10:      0.0282
Evaluated users: 574


Save the ranking results

In [17]:
ranking_output_path = (
    REPORT_DIR / "final_ranked_recommendations.csv"
)

ranked_recommendations.to_csv(
    ranking_output_path,
    index=False
)

ranking_metrics_df = pd.DataFrame({
    "Model": ["Random Forest Ranking"],
    "Recall@10": [ranking_recall],
    "Precision@10": [ranking_precision],
    "NDCG@10": [ranking_ndcg]
})

ranking_metrics_df.to_csv(
    REPORT_DIR / "ranking_model_metrics.csv",
    index=False
)

print("Ranked recommendations saved:")
print(ranking_output_path)

print("\nRanking metrics saved:")
print(REPORT_DIR / "ranking_model_metrics.csv")

Ranked recommendations saved:
c:\Users\yatha\Desktop\personalized-recommendation-engine\runs\reports\final_ranked_recommendations.csv

Ranking metrics saved:
c:\Users\yatha\Desktop\personalized-recommendation-engine\runs\reports\ranking_model_metrics.csv
